# Phase 5 - Evaluation: Car Price Prediction

Notebook ini menjalankan Phase 5 dari PRD dan ketentuan final project: mengevaluasi model Linear Regression menggunakan RMSE, R2 Score, akurasi berbasis R2, dan scatter plot actual vs predicted.

Input:
- `outputs/phase4/car_price_linear_regression_model.pkl`
- `outputs/phase3/X_test.csv`
- `outputs/phase3/y_test.csv`

Output:
- `outputs/phase5/evaluation_metrics.json`
- `outputs/phase5/evaluation_predictions.csv`
- `outputs/phase5/actual_vs_predicted_scatter.png`

## 1. Import Library

In [ ]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

## 2. Load Model dan Data Testing

Model yang digunakan adalah model hasil training pada Phase 4. Data testing berasal dari split 20% pada Phase 3.

In [ ]:
BASE_DIR = Path.cwd()
PHASE3_DIR = BASE_DIR / "outputs" / "phase3"
PHASE4_DIR = BASE_DIR / "outputs" / "phase4"
PHASE5_DIR = BASE_DIR / "outputs" / "phase5"
PHASE5_DIR.mkdir(parents=True, exist_ok=True)

model_path = PHASE4_DIR / "car_price_linear_regression_model.pkl"
model = joblib.load(model_path)

X_test = pd.read_csv(PHASE3_DIR / "X_test.csv")
y_test = pd.read_csv(PHASE3_DIR / "y_test.csv")["Price_in_thousands"]

print("Model:", type(model).__name__)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
X_test.head()

## 3. Prediksi Data Testing

In [ ]:
y_pred = model.predict(X_test)

predictions_df = X_test.copy()
predictions_df["Actual_Price_in_thousands"] = y_test.values
predictions_df["Predicted_Price_in_thousands"] = y_pred
predictions_df["Error"] = predictions_df["Actual_Price_in_thousands"] - predictions_df["Predicted_Price_in_thousands"]
predictions_df["Absolute_Error"] = predictions_df["Error"].abs()

predictions_df.head()

## 4. Evaluasi Model dengan RMSE dan R2 Score

RMSE mengukur rata-rata besar error prediksi dalam satuan ribuan USD. R2 Score menunjukkan proporsi variasi harga aktual yang dapat dijelaskan oleh model. Karena masalah ini adalah regresi, akurasi ditampilkan sebagai `R2 Score * 100%`.

In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)
accuracy_percent = r2 * 100

print(f"RMSE: {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")
print(f"Akurasi Model: {accuracy_percent:.2f}%")

## 5. Scatter Plot Actual vs Predicted

Titik yang semakin dekat dengan garis diagonal merah menunjukkan prediksi model yang semakin dekat dengan harga aktual.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.75, edgecolor="black")

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], color="red", linewidth=2, label="Ideal Prediction")

plt.title("Actual vs Predicted Car Price")
plt.xlabel("Actual Price (in thousands USD)")
plt.ylabel("Predicted Price (in thousands USD)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

scatter_plot_path = PHASE5_DIR / "actual_vs_predicted_scatter.png"
plt.savefig(scatter_plot_path, dpi=150)
plt.show()

print("Scatter plot saved to:", scatter_plot_path)

## 6. Simpan Hasil Evaluasi

In [ ]:
metrics = {
    "model_path": str(model_path),
    "test_rows": int(len(X_test)),
    "rmse": float(rmse),
    "r2_score": float(r2),
    "accuracy_percent": float(accuracy_percent),
    "scatter_plot_path": str(scatter_plot_path),
}

metrics_path = PHASE5_DIR / "evaluation_metrics.json"
predictions_path = PHASE5_DIR / "evaluation_predictions.csv"

metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
predictions_df.to_csv(predictions_path, index=False)

print("Evaluation metrics saved to:", metrics_path)
print("Evaluation predictions saved to:", predictions_path)

## 7. Analisis Singkat

Model menghasilkan nilai evaluasi yang cukup untuk baseline Linear Regression, tetapi R2 Score masih di bawah target PRD sebesar 0.80. Artinya, model baseline ini sudah dapat digunakan untuk demonstrasi pipeline CRISP-DM, namun peningkatan fitur atau pemilihan variabel tambahan dapat dipertimbangkan jika target akurasi harus dikejar.

## Phase 5 Checklist

- RMSE dihitung dan ditampilkan.
- R2 Score dihitung dan ditampilkan.
- Akurasi model ditampilkan sebagai persentase R2 Score.
- Scatter plot actual vs predicted dibuat.
- Metrik dan hasil prediksi disimpan untuk laporan akhir.